In [1]:
import torch
import torch.nn.functional as F
import pandas as pd
import glob
import os
from tqdm import tqdm


In [18]:
# 1. Setup paths
source_folder = "csv/*.csv" 
output_folder = "optimized_tensors" 
os.makedirs(output_folder, exist_ok=True)

# 2. Get file list
csv_files = sorted(glob.glob(source_folder))

print(f"Processing {len(csv_files)} files...")

for file_path in tqdm(csv_files):
    # --- A. Load Raw Data ---
    # Read the CSV (assuming no headers, if you have headers remove header=None)
    df = pd.read_csv(file_path, header=None) 
    
    # Convert to Tensor
    raw_tensor = torch.tensor(df.values, dtype=torch.float32)

    


    raw_tensor = raw_tensor[:,4:]
    raw_tensor = raw_tensor.view(3072, -1, 70)
    raw_tensor = raw_tensor.view(256, 12, -1, 70)
    raw_tensor = raw_tensor.mean(dim=1)
    raw_tensor = raw_tensor.permute(1, 2, 0).unsqueeze(1)
    raw_tensor = F.interpolate(raw_tensor , size=(244,244), mode='bicubic', align_corners=False)
    raw_tensor = raw_tensor.repeat(1, 3, 1, 1)
    raw_tensor = raw_tensor.permute(1, 2, 3, 0)

    raw_tensor = raw_tensor.permute(3, 0, 1, 2)


    # --- C. Save Smaller Tensor ---
    base_name = os.path.basename(file_path)
    new_name = base_name.replace('.csv', '.pt')
    save_path = os.path.join(output_folder, new_name)
    
    torch.save(raw_tensor, save_path)
    
    # Cleanup memory manually just to be safe with 500MB files
    del df, raw_tensor

print("Optimization Complete. Ready for training.")

Processing 2 files...


100%|██████████| 2/2 [00:09<00:00,  4.90s/it]

Optimization Complete. Ready for training.
